# Gradient Field and Descent Direction

Chapter 11: Matrix Calculus · *Mathematics for AI and Machine Learning* by Fuheng Wu

A live 3D demo: the gradient field on the floor under the bowl f; drag the start and compare gradient descent's zigzag with the heavy ball along the valley, per-step factors and the stability limit.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-11/gradient-field/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

# The demo's defaults: f(x, y) = 0.5 (a x^2 + b y^2), step size eta and the start point
a = 1.2
b = 2.2
eta = 0.2
start = torch.tensor([2.2, -1.8], dtype=torch.float64)


def f(p):
    return 0.5 * (a * p[0] ** 2 + b * p[1] ** 2)


# ── The gradient by hand and by autograd ────────────────────────────────────
p = start.clone().requires_grad_(True)
f(p).backward()
print("autograd grad f =", p.grad.tolist(), " by hand [a x, b y] =", [a * start[0].item(), b * start[1].item()])

# The gradient is the steepest ascent direction: compare with other unit directions
g = p.grad / p.grad.norm()
h = 1e-4
for name, d in (("-gradient", -g), ("x axis", torch.tensor([-1.0, 0.0], dtype=torch.float64)),
                ("y axis", torch.tensor([0.0, 1.0], dtype=torch.float64))):
    print(f"decrease along {name:10s}: {(f(p.detach()) - f(p.detach() + h * d)).item() / h:.4f}")

# ── Gradient descent x <- x - eta grad f ────────────────────────────────────
x = start.clone()
print("\n step   x          y          f")
for k in range(36):
    if k % 5 == 0:
        print(f"{k:4d}  {x[0].item():+.5f}  {x[1].item():+.5f}  {f(x).item():.6f}")
    x = x - eta * torch.stack([a * x[0], b * x[1]])

# Each coordinate shrinks by |1 - eta * curvature| per step
print("\nper-step factors: x", abs(1 - eta * a), " y", abs(1 - eta * b), "(both < 1 -> converges)")
print("largest stable step 2 / max(a, b) =", 2 / max(a, b))

autograd grad f = [2.64, -3.9600000000000004]  by hand [a x, b y] = [2.64, -3.9600000000000004]
decrease along -gradient : 4.7592
decrease along x axis    : 2.6399
decrease along y axis    : 3.9599

 step   x          y          f
   0  +2.20000  -1.80000  6.468000
   5  +0.55782  -0.09913  0.197505
  10  +0.14144  -0.00546  0.012035
  15  +0.03586  -0.00030  0.000772
  20  +0.00909  -0.00002  0.000050
  25  +0.00231  -0.00000  0.000003
  30  +0.00058  -0.00000  0.000000
  35  +0.00015  -0.00000  0.000000

per-step factors: x 0.76  y 0.5599999999999999 (both < 1 -> converges)
largest stable step 2 / max(a, b) = 0.9090909090909091
